# Pipeline làm sạch dữ liệu `tstd_HCM_3`

Áp dụng **cùng quy trình** với tập huấn luyện `train_HCM_3` cho tập test `tstd_HCM_3.xlsx` (710 dòng × 67 cột, toàn bộ là TSTĐ). Các rule dùng chung nằm trong `lam_sach.ipynb`.

**Đầu vào:**
- `data/tstd_HCM_3.xlsx` — dữ liệu test gốc
- `data/train_HCM_3.xlsx` — dữ liệu train gốc, chỉ để đối chiếu Mã tài sản ở bước cuối
- `output/bang_tra_he_so_hinh_dang.csv` — bảng tra hệ số hình dáng do notebook train tạo ra (chạy notebook train trước)

**Đầu ra** (ghi vào `output/`):
- `tstd_HCM_3_clean.xlsx` — dữ liệu test đã làm sạch
- `bao_cao_rule_tstd.csv` — số dòng tác động của từng rule
- `tstd_dong_bi_loai.csv` — các dòng bị loại vì trùng, kèm Mã tài sản của bản được giữ
- `tstd_tai_san_co_trong_train.csv` — Mã tài sản của test đã có trong train

Khác với train: file test **không có** hai cột `Số lượng mẫu` và `Tháng` nên chúng bị bỏ qua; các số đo chỉ được in ra, không có kỳ vọng cứng; bảng tra hệ số hình dáng **không** dựng lại từ dữ liệu test mà đọc từ file của tập train.

| Bước | Nội dung |
|---|---|
| B0 | Đọc dữ liệu gốc, chọn cột |
| B1 | Chuẩn hoá text và viết thường mọi cột chữ |
| B2 | Khử trùng lặp lần 1 (giữ bản có ngày hiệu lực muộn nhất) |
| B3 | Giữ dòng thiếu Đường phố |
| B4 | Làm sạch từng cột: số, Quận/Huyện, Đoạn đường, Yếu tố khác, Hình dáng, Chi tiết |
| B5 | Mâu thuẫn VT – KC – Chi tiết |
| B6 | Ô kích thước và cột tỷ lệ |
| B7 | Khử trùng lặp lần 2 (sau khi làm sạch) |
| B8 | Bổ sung ba cột chỉ số theo rule, tính trên dữ liệu gốc |
| B9 | Kiểm thử và xuất file |

## Cấu hình
Nạp các rule từ `lam_sach.ipynb`. Đầu vào đọc từ `data/`, đầu ra ghi vào `output/` của thư mục gốc dự án, nên notebook chạy được từ bất kỳ thư mục con nào.

In [1]:
from pathlib import Path

import pandas as pd

# Thư mục gốc dự án: đi lên từ thư mục đang chạy tới chỗ có 01_lam_sach/lam_sach.ipynb
GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '01_lam_sach' / 'lam_sach.ipynb').exists())
%run "{GOC / '01_lam_sach' / 'lam_sach.ipynb'}"

DATA = GOC / 'data' / 'raw'
OUTPUT = GOC / 'data' / 'processed'
OUTPUT.mkdir(exist_ok=True)

SRC = DATA / 'tstd_HCM_3.xlsx'
TRAIN_SRC = DATA / 'train_HCM_3.xlsx'
BANG_TRA = OUTPUT / 'bang_tra_he_so_hinh_dang.csv'     # do notebook train tạo ra
OUT = OUTPUT / 'tstd_HCM_3_clean.xlsx'
REPORT = OUTPUT / 'bao_cao_rule_tstd.csv'
LOAI = OUTPUT / 'tstd_dong_bi_loai.csv'
TRUNG_TRAIN = OUTPUT / 'tstd_tai_san_co_trong_train.csv'

bc = BaoCao()                                 # tập test không có kỳ vọng cứng: chỉ in số đo

## B0 · Đọc dữ liệu gốc và chọn cột
Giữ 14 cột làm sạch, các cột giữ nguyên nội dung có trong file (tập test thiếu `Số lượng mẫu` và `Tháng`), `Thành phố/Quận/Huyện/Thị xã`, `Mã tài sản` và target.

Các cột phụ (tên bắt đầu bằng `_`) chỉ dùng trong lúc xử lý và bị bỏ trước khi xuất:
- `_dong_goc`: số dòng trong file Excel gốc, để phá hoà khi chọn bản giữ;
- `_ngay_hieu_luc`: ngày hiệu lực, để giữ bản muộn nhất khi trùng;
- `_do_rong_goc`, `_dien_tich_goc`, `_hinh_dang_nguyen_van`: giá trị gốc chưa làm sạch, để B8 tính ba cột chỉ số đúng như công ty;
- `_cs_ngo_cty`, `_cs_dt_cty`, `_cs_hd_cty`: ba cột chỉ số có sẵn của công ty, chỉ để đối chiếu ở B8.

In [2]:
raw = doc_file_goc(SRC)
bc.kiem('B0_so_dong_goc', len(raw))
d = tao_bang_lam_viec(raw)
bc.kiem('B0_ngay_khong_doc_duoc', d['_ngay_hieu_luc'].isna().sum())
print(f'B0: {len(d)} dòng × {len(cot_xuat(d))} cột (chưa tính cột phụ)')

  B0_so_dong_goc = 710
Cột không có trong file này, bỏ qua: ['Số lượng mẫu', 'Tháng']
  B0_ngay_khong_doc_duoc = 0
B0: 710 dòng × 24 cột (chưa tính cột phụ)


## B1 · Chuẩn hoá text và viết thường
Áp cho **mọi cột chữ** (trừ cột phụ): chuẩn hoá Unicode NFKC, xoá ký tự vô hình, gộp khoảng trắng, viết thường; ô chỉ chứa một ký hiệu (`-`, `.`, `n/a`…) thành NaN.

Trên pandas 3, cột chữ có dtype `str` chứ không phải `object`, nên cột chữ được chọn theo điều kiện "không phải số và không phải ngày".

In [3]:
b1_chuan_hoa_text(d, bc)

  B1_so_cot_chu = 13


## B2 · Khử trùng lặp lần 1
- **Khoá:** 14 cột làm sạch + target. Không đưa cột thời gian hay cột nhóm vào khoá, vì cùng một tài sản thường được nhập lại ở nhiều tháng.
- **Chọn bản giữ:** `Thời điểm hiệu lực` muộn nhất; nếu hoà thì giữ dòng xuất hiện trước trong file gốc.

In [4]:
d, loai_b2 = khu_trung_lap(d, KHOA_TRUNG, 'B2')
bc.ghi('B2_khu_trung_lap', len(loai_b2), '14 cột làm sạch + target, giữ bản có ngày hiệu lực muộn nhất')

  [B2_khu_trung_lap]     17 dòng · 14 cột làm sạch + target, giữ bản có ngày hiệu lực muộn nhất


## B3 · Giữ dòng thiếu `Đường phố`
Không xoá. Khoá `Ghep` của các dòng này cũng thiếu tên đường nên không suy ngược được.

In [5]:
bc.kiem('B3_duong_pho_nan', d[DUONG].isna().sum())
bc.kiem('B3_so_dong', len(d))

  B3_duong_pho_nan = 0
  B3_so_dong = 693


## B4.1 · Cột số và Vị trí trong khung giá
| Rule | Lý do |
|---|---|
| `vt5` → `vt4` | VT5 quá ít dòng để tách riêng |
| Số mặt tiền = 0 hoặc > 4 → NaN | tối đa 4 mặt tiền; số 0 là ô bỏ trống |
| Độ rộng ngõ/ngách = 0 → NaN | đường nào cũng có bề rộng |
| Mặt tiền = 0 → NaN | thửa không thể có mặt tiền 0 m |
| KC > 2.500 m → NaN | nhập sai (ví dụ 350.150 m) |

In [6]:
b4_cot_so(d, bc)

  [B4_vt5_gop_vt4]      0 dòng · Vị trí trong khung giá
  [B4_smt_nan]      0 dòng · Số mặt tiền tiếp giáp
  [B4_do_rong_0_nan]      0 dòng · Độ rộng ngõ/ngách nhỏ nhất (Từ đường chính đến BĐS)
  [B4_mt_0_nan]      0 dòng · Kích thước mặt tiền (m)
  [B4_kc_qua_lon_nan]      0 dòng · Khoảng cách đến đường chính (m)


## B4.2 · Quận/Huyện
`thị xã thuận an` → `thành phố thuận an`: cùng một đơn vị có hai tên, nhập vào tên đang dùng ở các dòng khác.

In [7]:
b4_quan_huyen(d, bc)

  [B4_quan_huyen_doi_ten]      0 dòng · Thành phố/Quận/Huyện/Thị xã
  B4_quan_huyen_so_gia_tri = 32


## B4.3 · Đoạn đường
Gộp cách viết cùng nghĩa:
- `đầu đường - cuối đường`, `toàn tuyến` → `trọn đường`;
- cụm bắt đầu bằng `bề rộng mặt đường dưới` → `bề rộng mặt đường nhỏ hơn` (dạng khoảng `từ 6m đến dưới 9m` giữ nguyên);
- bỏ khoảng trắng giữa số và `m` (`9 m` → `9m`); bỏ dấu chấm cuối.

In [8]:
b4_doan_duong(d, bc)

  [B4_doan_duong_gop_cach_viet]      4 dòng · Đoạn đường
  B4_doan_duong_so_gia_tri = 223


## B4.4 · Yếu tố khác
Ô chỉ chứa số → NaN: `0` là ô bỏ trống, số lớn (ví dụ `394334534`) là số tiền nhập nhầm cột.

In [9]:
b4_yeu_to_khac(d, bc)

  [B4_yeu_to_khac_so_nan]      0 dòng · Yếu tố khác


## B4.5 · Hình dáng
Kết quả ghi vào cột mới `hinh_dang_sach`; cột `Hình dáng` gốc giữ nguyên.
1. **Số hoặc một chữ cái → NaN:** Hình dáng là chữ, nên số (`156.50`, `7,500,000`) và ký tự lẻ (`v`) là nhập nhầm.
2. **Sửa lỗi chính tả:** bảng tra lỗi gõ cố định (`cân dối`, `câm đối`, `cân đôi`… → `cân đối`), bỏ số dính vào chữ (`cân đối5`), đưa ba nhãn không phải hình dáng (`không`, `khá`, `nhà hẻm`) về NaN.
3. **Thống nhất cách viết:** `chữ l`, `hình l`, `(l)` → `hình chữ l`; các ý trong nhãn ghép cách nhau bằng `; ` (ví dụ `cân đối (nở hậu)` → `cân đối; nở hậu`). Không gom nhóm: mức độ (`nở hậu nhẹ`) và nhãn có số đo (`nở hậu 10m`) được giữ riêng.

In [10]:
b4_hinh_dang(d, bc)

  [B4_hinh_dang_so_nan]      0 dòng · hinh_dang_sach
  [B4_hinh_dang_sua_chinh_ta]      0 dòng · hinh_dang_sach
  [B4_hinh_dang_khong_phai_hinh_dang_nan]      0 dòng · hinh_dang_sach
  [B4_hinh_dang_thong_nhat_cach_viet]      0 dòng · hinh_dang_sach
  B4_hinh_dang_so_nhan = 3
  B4_hinh_dang_nan = 0


## B4.6 · Chi tiết
Hàm `tach_chi_tiet()` đổi một câu Chi tiết thành chuỗi các ý cách nhau bằng `; `:
1. **Thống nhất viết tắt:** `khu dân cư`, `kđc`, `khu dc` → `kdc`; `tđc`, `ktđc` → `khu tái định cư`; `kp` → `khu phố`; `kno` → `khu nhà ở`; `đnb`, `nội bộ` → `đường nội bộ`; `đs 2`, `đường 2` → `đường số 2`; `tl 37` → `tl37`.
2. **Ý vị trí đứng đầu:** `mt` (chữ mt, `2mt`, mặt tiền, mtnb, shophouse; hoặc mảnh đầu là số nhà đơn như `346`, dải số nhà `137-139`); `hẻm` (chữ hẻm; hoặc số nhà dạng `57/1a`, `102/` khi câu không có chữ mt).
3. **Các ý còn lại** theo thứ tự: `đường …` → `kdc …` → `khu …` → `ấp …` → phần còn lại.
4. **Bỏ:** mô tả nhà/đất (`nhà`, `đất`, `nhà nát`, `đất trống`, `biệt thự`, `có thang máy`…), số nhà, đường link, tên đường sau `mt`/`hẻm` khi trùng cột Đường phố.

Bản chữ trước khi tách được giữ tạm trong `_chi_tiet_b1` để B5 tính cờ và B7 so trùng.

In [11]:
b4_chi_tiet(d, bc)

  [B4_chi_tiet_tach_y]    585 dòng · Chi tiết
  B4_chi_tiet_so_gia_tri = 359
  B4_chi_tiet_nan = 46


## B5.1 · Bốn cờ nhận diện từ Chi tiết
Đọc trên `Chi tiết` **trước khi tách ý**. Chi tiết do chuyên viên ghi tay nên dùng làm trọng tài cho VT và KC.

| Cờ | Điều kiện |
|---|---|
| `la_hem` | có chữ `hẻm` |
| `co_mat_tien` | `\bmt\b`, `mặt tiền` hoặc `shophouse` |
| `chi_la_mat_tien` | chỉ ghi đúng `mt`, `mặt tiền`, `nhà mt`, `đất mt` |
| `la_so_nha` | chỉ là một số nhà, không có `/` |

`co_bang_chung_mat_tien = co_mat_tien | la_so_nha`.

In [12]:
b5_co_nhan_dien(d, bc)

  B5_la_hem = 2
  B5_co_mat_tien = 4
  B5_chi_la_mat_tien = 1
  B5_la_so_nha = 66


## B5.2 · Sửa 1: Chi tiết chỉ ghi "mt" thì là mặt tiền
Đặt `VT = vt1`, `KC = 0` cho `chi_la_mat_tien`, **trừ** khi VT là vt2–vt4 kèm KC cụ thể khác 0 (có thể là mặt tiền của đường ngoài bảng giá: chỉ gắn cờ ở B5.4). Đây là rule duy nhất sửa VT theo bằng chứng. Cờ `da_sua_vt_theo_mat_tien`.

In [13]:
b5_sua_mat_tien(d, bc)

  [B5_sua1_vt]      0 dòng · Vị trí trong khung giá
  [B5_sua1_kc]      0 dòng · Khoảng cách đến đường chính (m)
  B5_sua1_chi_gan_co = 0


## B5.3 · Sửa 2: số 0 ở KC phải có lý do
vt2–vt4 có `KC = 0` mà không có `co_bang_chung_mat_tien` → `KC = NaN` (số 0 nhiều khả năng là ô bỏ trống). Cờ `kc_thieu_du_lieu`.

In [14]:
b5_sua_kc_0(d, bc)

  [B5_sua2_kc_nan]      1 dòng · Khoảng cách đến đường chính (m)


## B5.4 · Hai cờ cảnh báo (chỉ gắn cờ, không sửa)
- `canh_bao_vt1`: VT1 nhưng có chữ hẻm, hoặc `KC > 0`, hoặc độ rộng < 4 m.
- `canh_bao_mt_vt234`: có chữ mt nhưng VT là vt2–vt4 và `KC > 0`.

In [15]:
b5_canh_bao(d, bc)

  [B5_canh_bao_vt1]      1 dòng · chỉ gắn cờ, không sửa
  [B5_canh_bao_mt_vt234]      0 dòng · chỉ gắn cờ, không sửa


## B6 · Ô kích thước và cột tỷ lệ
**Sửa 3:** `MT == DT` → `MT = DT / CD`; `CD == DT` → `CD = DT / MT`. Diện tích giữ nguyên vì là số trên giấy chứng nhận. Cờ `da_sua_kich_thuoc`.

Hai cột tỷ lệ chỉ để quan sát:
- `ty_le_kt_dt = MT × CD / DT`: thửa cân đối cho giá trị gần 1.
- `ty_le_dai_rong = CD / MT`: thửa ống có giá trị lớn.

In [16]:
b6_kich_thuoc(d, bc)

  [B6_sua3_mt_bang_dt]      0 dòng · Kích thước mặt tiền (m)
  [B6_sua3_cd_bang_dt]      0 dòng · Kích thước chiều dài


## B7 · Khử trùng lặp lần 2
Cùng cách chọn bản giữ như B2, chạy sau khi làm sạch vì các bước sửa có thể làm hai dòng trở nên giống hệt nhau.

Khoá giống B2 nhưng `Chi tiết` được so ở **dạng chữ trước khi tách ý** (`_chi_tiet_b1`): bộ tách bỏ số nhà, nên nếu so trên bản đã tách thì hai lô liền kề như `141` và `143` (cùng thành `mt`) sẽ bị gộp nhầm.

In [17]:
d, loai_b7 = khu_trung_lap(d, KHOA_B7, 'B7')
bc.ghi('B7_khu_trung_lap_lan_2', len(loai_b7), '14 cột + target sau khi làm sạch (Chi tiết so ở dạng chữ trước khi tách)')

  [B7_khu_trung_lap_lan_2]      0 dòng · 14 cột + target sau khi làm sạch (Chi tiết so ở dạng chữ trước khi tách)


## B8.1 · Bổ sung `Chi_so_ngo` và `Chi_so_dientich`
Hai cột này công ty **suy ra** từ cột khác (không phải nhập liệu), nên được bổ sung lại theo rule của công ty, tính trên **giá trị gốc** chưa làm sạch:

| Cột | Đầu vào | Rule (tính theo cận trên) |
|---|---|---|
| `Chi_so_ngo` | Độ rộng ngõ/ngách gốc (m) | ≤1 → 1; ≤1,5 → 2; ≤2 → 3; ≤2,5 → 4; ≤3 → 5; ≤3,5 → 6; ≤4 → 7; ≤5 → 8; ≤7 → 9; ≤10 → 10; ≤15 → 11; ≤25 → 12; trên 25 → 13 |
| `Chi_so_dientich` | Diện tích gốc (m²) | ≤30 → 1; ≤50 → 2; ≤70 → 3; ≤100 → 4; ≤150 → 5; ≤200 → 6; ≤300 → 7; trên 300 → 8 |

Sau đó đối chiếu với cột có sẵn của công ty.

In [18]:
b8_chi_so_ngo_dien_tich(d, bc)

  [B8_chi_so_ngo_khac_cong_ty]      0 dòng · Chi_so_ngo
  [B8_chi_so_dientich_khac_cong_ty]      0 dòng · Chi_so_dientich


## B8.2 · Bảng tra hệ số hình dáng (đọc từ tập train)
Rule của công ty: tra **nguyên văn** chuỗi Hình dáng trong bảng; tra được thì lấy hệ số, không tra được thì gán mặc định **0,95**.

Tập test dùng đúng bảng tra mà notebook train đã dựng từ dữ liệu gốc, không dựng lại từ dữ liệu test.

In [19]:
bang = doc_bang_tra(BANG_TRA)
bc.kiem('B8_bang_tra_so_chuoi', len(bang))
bc.kiem('B8_chuoi_hinh_dang_khong_co_trong_bang', (~d['_hinh_dang_nguyen_van'].isin(bang.index)).sum())

  B8_bang_tra_so_chuoi = 129
  B8_chuoi_hinh_dang_khong_co_trong_bang = 0


## B8.3 · Bổ sung `Chi_so_hinhdang`
Tra chuỗi Hình dáng **nguyên văn** của từng dòng trong bảng tra; không tra được thì gán 0,95. Sau đó đối chiếu với cột có sẵn của công ty: chỗ lệch là các dòng cũ mà công ty áp hệ số theo bảng trước khi cập nhật.

In [20]:
b8_chi_so_hinh_dang(d, bang, bc)

  [B8_chi_so_hinhdang_khac_cong_ty]      0 dòng · Chi_so_hinhdang
  khớp cột của công ty: 100.00% số dòng


## B9.1 · Kiểm thử
Kiểm các điều kiện của dữ liệu sạch trước khi xuất. Một điều kiện sai sẽ dừng notebook. `out` là bảng đầu ra, đã bỏ mọi cột phụ.

In [21]:
out = b9_kiem_thu(d, bc)

  B9_so_dong = 693
  B9_so_cot = 40
  B9_duong_pho_nan = 0
  OK không còn dòng trùng trên khoá B7


  OK không còn chữ hoa ở mọi cột chữ
  OK VT chỉ còn vt1–vt4
  OK số mặt tiền chỉ 1–4 hoặc NaN
  OK không còn 0 ở độ rộng và mặt tiền
  OK KC không vượt 2.500
  OK chi_la_mat_tien đúng rule (trừ dòng có canh_bao_mt_vt234)
  OK mọi dòng VT ≠ vt1 có KC = 0 đều có bằng chứng mặt tiền
  OK Chi tiết đúng định dạng ý; ý
  OK target > 0
  OK cột tỷ lệ không có inf
  OK Chi_so_ngo chỉ nhận 1–13, không NaN
  OK Chi_so_dientich chỉ nhận 1–8, không NaN
  OK Chi_so_hinhdang không NaN và nằm trong 0,70–1,05
  B9_so_dong_bi_sua = 1
Dòng bị sửa: 1 (0.14%)


## B9.2 · Xuất file
Lưu dữ liệu sạch, báo cáo rule và danh sách dòng bị loại vì trùng ở B2 và B7.

In [22]:
out.to_excel(OUT, index=False, sheet_name='tstd_clean')
bc.luu(REPORT)

bi_loai = pd.concat([loai_b2, loai_b7])
bi_loai = pd.DataFrame({
    'buoc_loai': bi_loai['buoc_loai'],
    'dong_excel_goc': bi_loai['_dong_goc'],
    'ma_tai_san': bi_loai[MA_TS],
    'ngay_hieu_luc': bi_loai['_ngay_hieu_luc'].dt.strftime('%d/%m/%Y'),
    'ma_ban_giu': bi_loai['ma_ban_giu'],
    **{c: bi_loai[c] for c in [PHUONG, DUONG, DT, TARGET]},
})
bi_loai['cung_ma_tai_san_voi_ban_giu'] = bi_loai['ma_tai_san'] == bi_loai['ma_ban_giu']
bi_loai.to_csv(LOAI, index=False, encoding='utf-8-sig')

print(f'Đã lưu {OUT.name}: {out.shape[0]} dòng × {out.shape[1]} cột')
print(f'Đã lưu {REPORT.name}: {len(bc.dong)} rule')
print(f'Đã lưu {LOAI.name}: {len(bi_loai)} dòng bị loại, '
      f'{int((~bi_loai["cung_ma_tai_san_voi_ban_giu"]).sum())} dòng có Mã tài sản khác bản được giữ')

Đã lưu tstd_HCM_3_clean.xlsx: 693 dòng × 40 cột
Đã lưu bao_cao_rule_tstd.csv: 25 rule
Đã lưu tstd_dong_bi_loai.csv: 17 dòng bị loại, 14 dòng có Mã tài sản khác bản được giữ


## B9.3 · Đối chiếu với tập train
- Mã tài sản của test (file gốc) đã từng được định giá trong train: so đơn giá test với đơn giá ở lần định giá gần nhất trong train (hoà ngày thì lấy dòng đứng trước trong file).
- Đếm dòng test giống hệt một dòng train trên 14 cột làm sạch + target (giá trị gốc).

Bước này đọc lại file train gốc nên mất khoảng 1–2 phút.

In [23]:
train = doc_file_goc(TRAIN_SRC)
train_gan_nhat = (train[[MA_TS, NGAY, TARGET]].dropna(subset=[MA_TS])
                  .assign(_ngay=lambda x: doc_ngay(x[NGAY]))
                  .sort_values('_ngay', ascending=False, kind='stable')
                  .drop_duplicates(MA_TS))
x = raw[[MA_TS, NGAY, TARGET]].merge(train_gan_nhat, on=MA_TS, suffixes=('_test', '_train'))
trung = pd.DataFrame({
    'ma_tai_san': x[MA_TS],
    'ngay_test': x[NGAY + '_test'],
    'gia_test': x[TARGET + '_test'],
    'ngay_train_gan_nhat': x[NGAY + '_train'],
    'gia_train': x[TARGET + '_train'],
})
trung['cung_gia'] = trung['gia_test'] == trung['gia_train']
trung.to_csv(TRUNG_TRAIN, index=False, encoding='utf-8-sig')
print(f'Đã lưu {TRUNG_TRAIN.name}: {len(trung)} Mã tài sản có trong train, {int(trung["cung_gia"].sum())} trường hợp cùng đơn giá')

giong_het = raw[KHOA_TRUNG].astype(str).merge(train[KHOA_TRUNG].astype(str).drop_duplicates())
bc.kiem('B9_dong_test_giong_het_train', len(giong_het))

Đã lưu tstd_tai_san_co_trong_train.csv: 69 Mã tài sản có trong train, 23 trường hợp cùng đơn giá
  B9_dong_test_giong_het_train = 0
